# APTOS size-stratified check
**Question:** is the high APTOS score of the DDR-trained models (AUROC ~0.99) explained by image **size**? APTOS images of different sizes come from different cameras and clinics, and size alone separates early DR from No DR with AUROC 0.96 there.

**Method:** inside a group of APTOS images that all have the *same size*, size cannot help, so the model's AUROC within such groups shows how much of the score is real. No model is run and no test set is opened again: it uses the saved `preds_APTOS.csv` files and image sizes. This analysis is **post-hoc** and is labelled that way.

**Before running:** add (1) the output of your original run (`manifest.csv`, `runs/`) and (2) the output of `image_property_check` (it has `audit/image_features.csv` with every image size). If you cannot, add the APTOS dataset instead. The notebook stops with a message if it cannot read the sizes. CPU is enough; takes a minute or two.

**What to send me:** the final table and the verdict lines.

In [ ]:
import base64, glob, json, os, shutil, subprocess, sys, zlib
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("EARLYDR_WORK", "/kaggle/working")
CODE = os.path.join(WORK, "code")
EXTRA = os.environ.get("EARLYDR_TEST", "").split()     # test hook, leave empty on Kaggle
CODE_FILES = (
    "..."
    "j68/"
)
try:
    import timm  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"])
for name, text in json.loads(zlib.decompress(base64.b64decode(CODE_FILES))).items():
    path = os.path.join(CODE, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(text)
print("code written to", CODE)
sys.path.insert(0, CODE)

def real_dirs(pattern):
    return [os.path.dirname(f) for f in glob.glob(os.path.join(INPUT, "**", pattern), recursive=True)]

def processed_in(folder):
    return [p for p in glob.glob(os.path.join(folder, "processed_*")) if os.path.isdir(p) and not os.path.islink(p)]

dirs = [d for d in real_dirs("manifest.csv") if glob.glob(os.path.join(d, "runs", "*", "seed*", "preds_APTOS.csv"))]
if not dirs:
    sys.exit("STOP: add the output of your original run (manifest.csv + runs/*/seed*/preds_APTOS.csv) as an input.")
DATA = dirs[0]
print("using", DATA)
caches = glob.glob(os.path.join(INPUT, "**", "audit", "image_features.csv"), recursive=True)
cache = caches[0] if caches else None
print("cached image sizes:", cache)
arms = sorted(os.path.basename(p) for p in glob.glob(os.path.join(DATA, "runs", "*")) if glob.glob(os.path.join(p, "seed*", "preds_APTOS.csv")))
print("arms with APTOS predictions:", arms)

## Compare overall and within-same-size AUROC

In [ ]:
from early_dr import size_strata as S
table, sizes = S.analyse(DATA, arms, sizes_cache=cache, search_root=INPUT)
display(table)
print(S.verdict(table))
display(sizes.head(15))
os.makedirs(os.path.join(WORK, "audit"), exist_ok=True)
table.to_csv(os.path.join(WORK, "audit", "aptos_size_strata.csv"), index=False)
sizes.to_csv(os.path.join(WORK, "audit", "aptos_size_groups.csv"))

## How to read it
* **overall, usable sizes** = AUROC on the images in size groups big enough to compare; **within same size** = AUROC using only pairs from the same size group.
* About equal -> the score does **not** come from image size. A fall of 0.02 or more -> part of the score may come from size.
* Images in small size groups are left out of both numbers, so they stay comparable.